# Report generation

PyGSTi constructs polished report documents that give both high-level summaries and detailed analyses of results, gate set tomography (GST) and model-testing results in particular.  Reports are meant to be a quick and easy way of analyzing `Model`-type estimates, and pyGSTi's report generation functions are designed to work with the `ModelEstimateResults` object produced by pyGSTi's GST protocols (see, for example, the [GST overview](../../start/FirstGST)).  A report generation function takes one or more results objects as input and produces an HTML file as output.  The HTML format lets reports include **interactive plots** and **switches** (see the [workspace switchboard guide](../../advanced/figures/Switchboards)), which makes it easy to compare different types of analysis or different data sets.

PyGSTi's reports are stand-alone HTML documents that cannot run Python.  Everything displayed in a report is pre-computed.  If you find yourself wanting to fiddle with things and feel that these reports are too static, use a `Workspace` object (see [Workspace tables and plots](../../advanced/figures/WorkspaceFigures)) inside a Jupyter notebook, where you can intermix report tables/plots and Python.  Internally, functions like `construct_standard_report` are simple factories for `Report` objects, which are in turn little more than a wrapper around a `Workspace` object plus a set of instructions for how to generate output in different formats.

Every report the example notebooks generate is served with these docs, so you can read one without running anything first. Each is linked from the cell that writes it, and they are all listed together at <a href="../../../reports/">example reports</a>.

## Get some `ModelEstimateResults`

Start by performing GST to create a `ModelEstimateResults` object (you could also just load one from file).  The calls below use the protocol-object API — an experiment design paired with data and run through a protocol — described in [migrating from the function-based API](../../advanced/migration/FromFunctionAPI).

In [1]:
import pygsti
from pygsti.modelpacks import smq1Q_XYI

target_model = smq1Q_XYI.target_model()
prep_fiducials = smq1Q_XYI.prep_fiducials()
meas_fiducials = smq1Q_XYI.meas_fiducials()
germs = smq1Q_XYI.germs()
maxLengths = [1,2,4,8,16]
ds = pygsti.io.read_dataset("../../../tutorial_files/Example_Dataset.txt", cache=True)

#Run GST
target_model.set_all_parameterizations("full TP") #TP-constrained
edesign = pygsti.protocols.StandardGSTDesign(target_model.create_processor_spec(), prep_fiducials,
                                             meas_fiducials, germs, maxLengths)
data = pygsti.protocols.ProtocolData(edesign, ds)

results = pygsti.protocols.GateSetTomography(target_model, verbosity=3).run(data)

Reading from cache file: ../../../tutorial_files/Example_Dataset.txt.cache
  Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
    Using MapForwardSimulator without MPI
    Using MapForwardSimulator without MPI
    Using MapForwardSimulator without MPI
    Using MapForwardSimulator without MPI
    Using MapForwardSimulator without MPI
--- Iterative GST: Iter 1 of 5  92 circuits ---: 
  --- chi2 GST ---
    --- Outer Iter 0: norm_f = 1.10824e+07, mu=1, |x|=3.24037, |J|=35370.4
    --- Outer Iter 1: norm_f = 320.95, mu=50045, |x|=3.06272, |J|=1128.3
    --- Outer Iter 2: norm_f = 137.654, mu=16681.7, |x|=3.04438, |J|=1040.55
    --- Outer Iter 3: norm_f = 95.8691, mu=5560.55, |x|=3.03114, |J|=1029.04
    --- Outer Iter 4: norm_f = 69.5263, mu=1853.52, |x|=3.01363, |J|=1046.83
    --- Outer Iter 5: norm_f = 63.1005, mu=4942.71, |x|=3.00808, |J|=1119.82
    --- Outer Iter 6: norm_f = 61.2404, mu=9885.3, |x|=3.00222, |J|=4005.81
    --- Outer Iter 7: norm_f = 57.6458, 

## Make a report

Now that we have `results`, use `construct_standard_report` within `pygsti.report` to generate a `Report`.  `pygsti.report.construct_standard_report` is the most commonly used report factory function in pyGSTi; it's appropriate for smaller models (1- and 2-qubit) whose *operations are, or can be represented as, dense matrices and/or vectors*.

Once constructed, a `Report` object can write itself out as an HTML document, a PDF, or a notebook.  To open an HTML-format report, open the `main.html` file inside the report's folder.  Setting `auto_open=True` makes the finished report open in your web browser automatically.

In [2]:
report = pygsti.report.construct_standard_report(results, title="GST Example Report", verbosity=1)
#HTML
report.write_html("../../../tutorial_files/exampleReport", connected=True, auto_open=False, verbosity=1)

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XYI


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleReport.html">exampleReport</a>.

The `connected` argument decides where a report's JavaScript and CSS come from.  At the default `connected=False`, pyGSTi copies an `offline` folder of libraries (jQuery, Plotly, KaTeX) in beside `main.html`; that adds about 9 MB to every report and lets it render on a machine with no network connection.  Passing `connected=True` loads the same libraries from a CDN instead, leaving the report as a single `main.html` file that needs a network connection to draw its figures.  Every report in this documentation is written with `connected=True`, because they are served from a web site whose readers are online by definition.  For a report you mean to email to a colleague or keep for the archive, leave the default.

In [ ]:
#PDF
report.write_pdf("../../../tutorial_files/exampleReport.pdf", auto_open=False, verbosity=1)

Several remarks about these reports are worth noting:

1. The **HTML reports are the primary report type in pyGSTi**, and are much more flexible.  The PDF reports are more limited (they can only display a *single* estimate and gauge optimization), and essentially contain a subset of the information and descriptive text of an HTML report.  So, if you can, use the HTML reports.  The PDF report's strength is its portability: PDFs are easily displayed by many devices, and they embed all that they need neatly into a single file.  **If you need to generate a PDF report** from `Results` objects that have multiple estimates and/or gauge optimizations, consider using the `Results` object's `view` method to single out the estimate and gauge optimization you're after.
2. It's best to use **Firefox** when opening the HTML reports.  (If there's a problem with your browser's capabilities it will be shown on the screen when you try to load the report.)
3. You'll need **`pdflatex`** on your system to compile PDF reports.
4. To familiarize yourself with the layout of an HTML report, click on the gray **"Help" link** on the black sidebar.

## Multiple estimates in a single report

Next, analyze the same data two different ways: with and without the TP constraint (that is, whether the gates *must* be trace-preserving), gauge optimizing each case using several different SPAM weights.  In each case we run `GateSetTomography` with `gaugeopt_suite=None`, so that no gauge optimization is done, then perform several gauge optimizations separately and add these to the `Results` object via its `add_gaugeoptimized` method.  Each run gets a distinct `name` so the two estimates can later live side by side in a single `Results` object.  Both cases fit the same circuits collected in `ds`, so both reuse the `edesign`/`data` pair built above.

In [3]:
#Case1: TP-constrained GST
tpTarget = target_model.copy()
tpTarget.set_all_parameterizations("full TP")
results_tp = pygsti.protocols.GateSetTomography(tpTarget, gaugeopt_suite=None, name='full TP',
                                               verbosity=1).run(data)

#Gauge optimize
est = results_tp.estimates['full TP']
mdlFinal = est.models['final iteration estimate']
mdlTarget = est.models['target']
for spamWt in [1e-4,1e-2,1.0]:
    mdl = pygsti.gaugeopt_to_target(mdlFinal,mdlTarget,{'gates':1, 'spam':spamWt})
    est.add_gaugeoptimized({'item_weights': {'gates':1, 'spam':spamWt}}, mdl, "Spam %g" % spamWt)

--- Iterative GST: [##################################################] 100.0%  616 circuits ---


<repo>/pygsti/algorithms/gaugeopt.py:225: DeprecatedPositionalArgumentsWarning: 
            Recieved True positional arguments past `model` and `target_model`.
            These should be passed as appropriate keyword arguments instead. This version
            of pyGSTi will infer intended keyword arguments based on the legacy argument 
            positions. Future versions of pyGSTi will raise an error.
            
  _warnings.warn(msg, DeprecatedPositionalArgumentsWarning)


In [4]:
#Case2: "Full" GST
fullTarget = target_model.copy()
fullTarget.set_all_parameterizations("full")
results_full = pygsti.protocols.GateSetTomography(fullTarget, gaugeopt_suite=None, name='Full',
                                                 verbosity=1).run(data)

#Gauge optimize
est = results_full.estimates['Full']
mdlFinal = est.models['final iteration estimate']
mdlTarget = est.models['target']
for spamWt in [1e-4,1e-2,1.0]:
    mdl = pygsti.gaugeopt_to_target(mdlFinal,mdlTarget,{'gates':1, 'spam':spamWt})
    est.add_gaugeoptimized({'item_weights': {'gates':1, 'spam':spamWt}}, mdl, "Spam %g" % spamWt)

--- Iterative GST: [##################################################] 100.0%  616 circuits ---


Now call the *same* `construct_standard_report` function, but instead of passing a single `Results` object as the first argument pass a *dictionary* of them.  The result is an **HTML report that includes switches** for selecting which case ("TP" or "Full") and which gauge optimization to display output quantities for.  PDF reports cannot support this interactivity, so **if you try to generate a PDF report you'll get an error**.

In [5]:
ws = pygsti.report.Workspace()
report = pygsti.report.construct_standard_report(
    {'full TP': results_tp, "Full": results_full}, title="Example Multi-Estimate Report", ws=ws, verbosity=2)
report.write_html("../../../tutorial_files/exampleMultiEstimateReport", connected=True, auto_open=False, verbosity=2)

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI
Comparing data sets 'Full' and 'Full':
Statistical hypothesis tests did NOT find inconsistency between the data at 5.00% significance.
Comparing data sets 'Full' and 'full TP':
Statistical hypothesis tests did NOT find inconsistency between the data at 5.00% significance.
Comparing data sets 'full TP' and 'Full':
Statistical hypothesis tests did NOT find inconsistency between the data at 5.00% significance.
Comparing data sets 'full TP' and 'full TP':
Statistical hypothesis tests did NOT find inconsistency between the data at 5.00% significance.


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleMultiEstimateReport.html">exampleMultiEstimateReport</a>.

The call above constructs `ws`, a `Workspace` object.  PyGSTi's `Workspace` objects are both a factory for figures and tables and a smart cache for computed values.  A `Workspace` object can optionally be passed to `construct_standard_report`, where it is used to create all the figures in the report.  As an intended side effect, each of those figures is cached, along with some of the intermediate results used to create it.  Passing a preconstructed `Workspace` object to `construct_standard_report` lets it reuse previously cached quantities.

**Another way**: because `results_tp` and `results_full` used the same dataset and operation sequences, they could have been combined as two estimates in a single `ModelEstimateResults` object (see [Results](Results) for the structure of those objects).  Add the estimate within `results_full` to the estimates already contained in `results_tp`:

In [6]:
results_both = results_tp.copy() #copy just for neatness
results_both.add_estimates(results_full, estimates_to_add=['Full'])

<repo>/pygsti/protocols/gst.py:3164: UserWarning: Provided estimate Full has different parent than `self`.
We'll make a copy of this estimate and set its parent to `self`.
  _warnings.warn(msg)


Creating a report from `results_both` gives the same report we just generated.  We'll do it anyway, this time supplying `construct_standard_report` with the same `Workspace` used before.  That tells the constructed `Report` to use any cached values in the given *input* `Workspace` to expedite report generation.  Since our workspace has exactly the quantities we need cached in it, you'll notice a significant speedup.  Note that even though there's just a single `Results` object, you **still can't generate a PDF report** from it, because it contains multiple estimates.

In [7]:
pygsti.report.construct_standard_report(
    results_both,
    title="Example Multi-Estimate Report (v2)", 
    ws=ws, verbosity=2
).write_html("../../../tutorial_files/exampleMultiEstimateReport2", connected=True, auto_open=False, verbosity=2)

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleMultiEstimateReport2.html">exampleMultiEstimateReport2</a>.

## Multiple estimates and `StandardGST`

It's no coincidence that a `Results` object containing multiple estimates from the same data is precisely what the `StandardGST` protocol returns.  It runs GST several times, creating different "standard" estimates and gauge optimizations, so you can plot them all in a single HTML report.

In [8]:
results_std = pygsti.protocols.StandardGST(modes=('full TP', 'CPTPLND', 'Target'),
                                           gaugeopt_suite=('stdgaugeopt','toggleValidSpam'),
                                           target_model=target_model, verbosity=4).run(data)

# Generate a report with "TP", "CPTP", and "Target" estimates
pygsti.report.construct_standard_report(
    results_std, title="Post StdPractice Report", verbosity=1
).write_html("../../../tutorial_files/exampleStdReport", connected=True, auto_open=False, verbosity=1)

-- Std Practice:  Iter 1 of 3  (full TP) --: 
    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
      Using MapForwardSimulator without MPI
  --- Iterative GST: Iter 1 of 5  92 circuits ---: 
    --- chi2 GST ---
      --- Outer Iter 0: norm_f = 1.10824e+07, mu=1, |x|=3.24037, |J|=35370.4
      --- Outer Iter 1: norm_f = 320.95, mu=50045, |x|=3.06272, |J|=1128.3
      --- Outer Iter 2: norm_f = 137.654, mu=16681.7, |x|=3.04438, |J|=1040.55
      --- Outer Iter 3: norm_f = 95.8691, mu=5560.55, |x|=3.03114, |J|=1029.04
      --- Outer Iter 4: norm_f = 69.5263, mu=1853.52, |x|=3.01363, |J|=1046.83
      --- Outer Iter 5: norm_f = 63.1005, mu=4942.71, |x|=3.00808, |J|=1119.82
      --- Outer Iter 6: norm_f = 61.2404, mu=9885.3, |x|=3.00222, |J|=4005.81
      --- Outer Iter 7: norm_f = 57.645

      --- Outer Iter 0: norm_f = 170.051, mu=1, |x|=0.488619, |J|=1456.83
      --- Outer Iter 1: norm_f = 136.047, mu=339.186, |x|=0.505981, |J|=1415.82
      --- Outer Iter 2: norm_f = 124.041, mu=142.313, |x|=0.500827, |J|=1429.26
      --- Outer Iter 3: norm_f = 122.853, mu=134.178, |x|=0.499083, |J|=1432.8
      --- Outer Iter 4: norm_f = 122.489, mu=133.867, |x|=0.499171, |J|=1434.97
      --- Outer Iter 5: norm_f = 122.369, mu=1047.44, |x|=0.498816, |J|=1436.13
      --- Outer Iter 6: norm_f = 122.362, mu=1127.19, |x|=0.49885, |J|=1436.17
      --- Outer Iter 7: norm_f = 122.358, mu=1311.83, |x|=0.498939, |J|=1436.1
      --- Outer Iter 8: norm_f = 122.355, mu=1342.43, |x|=0.499014, |J|=1436.03
      --- Outer Iter 9: norm_f = 122.352, mu=1342.41, |x|=0.499088, |J|=1435.96
      --- Outer Iter 10: norm_f = 122.35, mu=1249.62, |x|=0.499164, |J|=1435.88
      --- Outer Iter 11: norm_f = 122.348, mu=838.716, |x|=0.499246, |J|=1435.79
      --- Outer Iter 12: norm_f = 122.343, mu=27

<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleStdReport.html">exampleStdReport</a>.

## Reports with confidence regions

To display confidence intervals for reported quantities, you must do two things:

1. specify the `confidence_level` argument to `construct_standard_report`.
2. give the estimate(s) being reported a valid confidence-region factory.

Constructing a factory often means computing a Hessian, which can be time consuming, so it isn't done automatically.  Here's how to construct a valid factory for the "Spam 0.001" gauge optimization of the "CPTP" estimate, by computing and then projecting the Hessian of the likelihood function.

In [9]:
#Construct and initialize a "confidence region factory" for the CPTP estimate
crfact = results_std.estimates["CPTPLND"].add_confidence_region_factory('Spam 0.001', 'final')
crfact.compute_hessian(comm=None) #we could use more processors
crfact.project_hessian('intrinsic error')

pygsti.report.construct_standard_report(
    results_std, title="Post StdPractice Report (w/CIs on CPTP)",
    confidence_level=95, verbosity=1
).write_html("../../../tutorial_files/exampleStdReport2", connected=True, auto_open=False, verbosity=1)

    
--- Hessian Projector Optimization from separate SPAM and Gate weighting ---
  Resulting intrinsic errors: 190468 (gates), 821600 (spam)
  Resulting sqrt(mean(operationCIs**2)): 199261
  Resulting sqrt(mean(spamCIs**2)): 150121
Running idle tomography
Computing switchable properties


<repo>/pygsti/report/factory.py:1373: UserWarning: confidence_level=95 requests error bars, but no confidence region factory with a computed Hessian was found for estimate(s) ['full TP']. These estimates will be rendered without error bars. Call ModelEstimateResults.add_hessians() on your results object(s) before report generation to compute the necessary Hessians.
  _warnings.warn(


Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleStdReport2.html">exampleStdReport2</a>.

## Reports with multiple *different* data sets

We've already seen that `construct_standard_report` can be given a dictionary of `Results` objects instead of a single one.  That also allows reports containing estimates for different `DataSet`s, since each `Results` object only holds estimates for a single `DataSet`.  When the data sets have the same operation sequences, they're compared within a tab of the HTML report.

Below, we generate a new data set with the same sequences as the one loaded at the beginning of this page, run standard-practice GST on it, and create a report of those results alongside the original data set's.  Look at the **"Data Comparison" tab** within the gauge-invariant error metrics category.

In [10]:
#Make another dataset & estimates
target_model = smq1Q_XYI.target_model('full TP')
depol_gateset = target_model.depolarize(op_noise=0.1)
datagen_gateset = depol_gateset.rotate((0.05,0,0.03))

#Compute the sequences needed to perform long-sequence GST on this Model,
# using the same maxLengths as the fit below so the data covers what GST asks for
circuit_list = pygsti.circuits.create_lsgst_circuits(
    smq1Q_XYI.target_model(), smq1Q_XYI.prep_fiducials(), smq1Q_XYI.meas_fiducials(),
    smq1Q_XYI.germs(), maxLengths)
ds2 = pygsti.data.simulate_data(datagen_gateset, circuit_list, num_samples=1000,
                                             sample_error='binomial', seed=2018)

#Same circuits as `edesign`, just paired with the new dataset
data2 = pygsti.protocols.ProtocolData(edesign, ds2)
results_std2 = pygsti.protocols.StandardGST(modes=('full TP', 'Target'),
                                            gaugeopt_suite=('stdgaugeopt','toggleValidSpam'),
                                            target_model=target_model, verbosity=3).run(data2)

pygsti.report.construct_standard_report(
    {'DS1': results_std, 'DS2': results_std2},
    title="Example Multi-Dataset Report", verbosity=1
).write_html("../../../tutorial_files/exampleMultiDataSetReport", connected=True, auto_open=False, verbosity=1)

-- Std Practice:  Iter 1 of 2  (full TP) --: 
    Precomputing CircuitOutcomeProbabilityArray layouts for each iteration.
  --- Iterative GST: Iter 1 of 5  92 circuits ---: 
    --- chi2 GST ---
    Sum of Chi^2 = 50.2567 (92 data params - 43 (approx) model params = expected mean of 49; p-value = 0.423422)
    Completed in 0.3s
    Iteration 1 took 1.1s
    
  --- Iterative GST: Iter 2 of 5  168 circuits ---: 
    --- chi2 GST ---
    Sum of Chi^2 = 112.85 (168 data params - 43 (approx) model params = expected mean of 125; p-value = 0.774068)
    Completed in 0.4s
    Iteration 2 took 0.4s
    
  --- Iterative GST: Iter 3 of 5  285 circuits ---: 
    --- chi2 GST ---
    Sum of Chi^2 = 222.419 (285 data params - 43 (approx) model params = expected mean of 242; p-value = 0.811854)
    Completed in 0.6s
    Iteration 3 took 0.6s
    
  --- Iterative GST: Iter 4 of 5  448 circuits ---: 
    --- chi2 GST ---
    Sum of Chi^2 = 405.964 (448 data params - 43 (approx) model params = expected 

Running idle tomography
Computing switchable properties


<repo>/pygsti/report/factory.py:1465: UserWarning: Not all data sets are comparable - no comparisions will be made.
  _warnings.warn("Not all data sets are comparable - no comparisions will be made.")


Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleMultiDataSetReport.html">exampleMultiDataSetReport</a>.

## Reports from LGST alone

Reports aren't restricted to long-sequence GST.  *Linear* GST (LGST) takes substantially less data and computation time, so when a rough estimate of your gates is all you're after, it's worth knowing that its output feeds the same report machinery.  The experiment design below uses `max_max_length=1`, which is all LGST requires.

This workflow differs from the ones above in how the data arrives: instead of pairing an existing `DataSet` with an experiment design, write an empty data directory from the experiment design, fill it in (standing in for actually collecting data), read the completed directory back, then run the protocol.

In [11]:
#Get experiment design (for now, just max_max_length=1 GST sequences)
exp_design = smq1Q_XYI.create_gst_experiment_design(max_max_length=1)
pygsti.io.write_empty_protocol_data("../../../example_files/lgst_only_example", exp_design, clobber_ok=True)
print("Only %d sequences are required!" % len(exp_design.all_circuits_needing_data))

#Simulate taking the data (here you'd really fill in dataset.txt with actual data)
mdl_datagen = smq1Q_XYI.target_model().depolarize(op_noise=0.1, spam_noise=0.001)
pygsti.io.fill_in_empty_dataset_with_fake_data("../../../example_files/lgst_only_example/data/dataset.txt",
                                               mdl_datagen, num_samples=1000, seed=2020)

#load in the data
lgst_data = pygsti.io.read_data_from_dir("../../../example_files/lgst_only_example")

Only 92 sequences are required!


In [12]:
#Run LGST on the data written above.
results_lgst = pygsti.protocols.LGST(smq1Q_XYI.target_model()).run(lgst_data)

--- LGST ---
  Singular values of I_tilde (truncating to first 4 of 6) = 
  4.243739384333438
  1.174085597747583
  0.9927915010167848
  0.9182042113995429
  0.07317359677984964
  0.032353259963406794
  
  Singular values of target I_tilde (truncating to first 4 of 6) = 
  4.242640687119286
  1.4142135623730956
  1.4142135623730954
  1.4142135623730951
  2.737523835737296e-16
  2.1099286572735318e-16
  
  Using MapForwardSimulator without MPI


In [13]:
pygsti.report.construct_standard_report(
    results_lgst, title="LGST-only Example Report", verbosity=2
).write_html('../../../example_files/LGSTonlyReport', connected=True, auto_open=False, verbosity=2)

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XYI


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


That report is served with these docs: <a href="../../../reports/LGSTonlyReport.html">LGSTonlyReport</a>.

## Other `Report` tricks

A few additional arguments to the `Report` output methods give further control over what ends up in the generated report.

- Setting the `link_to` argument to a tuple of `'pkl'`, `'tex'`, and/or `'pdf'` creates hyperlinks within the plots or below the tables of the HTML, pointing at Python pickle, LaTeX source, and PDF versions of the content.  The pickle files for tables contain pickled pandas `DataFrame` objects; those for plots contain ordinary Python dictionaries of the plotted data.  Applies to HTML reports only.

- Setting the `brevity` argument to an integer higher than $0$ (the default) reduces the amount of information included in the report (for what's included at each value, see the doc string).  Using `brevity > 0` cuts the time required to create, and later load, the report, along with the output file/folder size.  This applies to both HTML and PDF reports.

Below we demonstrate both options in a very brief (`brevity=4`) report with links to pickle and LaTeX files.  Note that generating `'pdf'` links requires `pdflatex`.

In [14]:
pygsti.report.construct_standard_report(
    results_std, title="Example Brief Report", verbosity=1
).write_html("../../../tutorial_files/exampleBriefReport", connected=True, auto_open=False, verbosity=1,
             brevity=4, link_to=('pkl', 'tex'))

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI
Found standard clifford compilation from smq1Q_XYI


<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(
<env>/lib/python3.13/site-packages/plotly/offline/offline.py:150: UserWarning: 
Unrecognized config options supplied: ['showLink', 'linkText']
  warnings.warn(


Served with these docs: <a href="../../../reports/exampleBriefReport/main.html">exampleBriefReport</a>.

## Report notebooks: `Report.write_notebook`

Besides the standard HTML-page reports demonstrated above, pyGSTi can generate a Jupyter notebook containing the Python commands that create the figures and tables of a general report.  `Workspace` objects, being factories for figures and tables, make this possible.  Calling `Report.write_notebook` dumps all of the relevant `Workspace` initialization and calls to a new notebook file, which you can then run fully or partially at your convenience.  The advantage is that you can insert Python code amidst the figure and table generation calls to inspect or modify what's displayed.  The disadvantages: report notebooks require a running Jupyter server, and nothing is displayed until you run the notebook.

```{note}
Interactive cells in report notebooks are driven by JavaScript, so a report notebook has to be "Trusted" before its figures will appear: JupyterLab and Notebook 7 refuse to run scripts in an untrusted notebook.  Run `jupyter trust yournotebook.ipynb`, or use the Trusted indicator in the toolbar.  Older versions of pyGSTi could not render these cells under JupyterLab at all (https://github.com/pyGSTio/pyGSTi/issues/205); that limitation is gone, and classic Jupyter Notebook is no longer required.
```

The line below creates a report notebook with `write_notebook`.  The argument list is very similar to the other `Report` output methods.

In [15]:
pygsti.report.construct_standard_report(
    results, title="GST Example Report Notebook", confidence_level=None, verbosity=3
).write_notebook("../../../tutorial_files/exampleReport.ipynb", auto_open=False, connected=False, verbosity=3)

Running idle tomography
Computing switchable properties
Found standard clifford compilation from smq1Q_XYI
Report Notebook created as ../../../tutorial_files/exampleReport.ipynb
Note: the figures in a report notebook are drawn by JavaScript, so the notebook must be "Trusted" before they will render.


## Multi-qubit reports

The density matrix space of more than 2 qubits gets quite large, and models for 3+ qubits rarely let every element of the operation process matrices vary independently.  Many of the figures generated by `construct_standard_report` are therefore both unwieldy (a $64 \times 64$ grid of colored boxes for each operation) and unhelpful (you don't often care what each element of an operation matrix is).  For this case we are developing a report that doesn't just dump out and analyze operation matrices as a whole, but looks at a `Model`'s structure to decide how best to report quantities.  This "n-qubit report" is invoked using `pygsti.report.construct_nqnoise_report`, and takes arguments similar to `construct_standard_report`.  It is, however, <b style="color:red">still under development</b>, and while you're welcome to try it out, it may crash or fail in other weird ways.